In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from src.to_html import cell, heading, paragraph, table

In [2]:
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[[
    "delay_in_min", "train_number", "train_type",
    "arrival_is_canceled", "departure_is_canceled",
    "is_replacement_train", "replaced_train_number",
    ]])
df = pd.concat(df_list, ignore_index=True)

# Train numbers whose scheduled service was provided by a replacement train
# (t="e"). Those stops are not counted as cancelled because the service still
# ran, just under a different train number.
replaced_numbers = set(df.loc[df["is_replacement_train"], "replaced_train_number"].dropna())
df["has_replacement"] = df["train_number"].isin(replaced_numbers)
# ran = the stop was actually served (includes additional stops and replacement trains)
df["ran"] = ~(df["arrival_is_canceled"] | df["departure_is_canceled"])
# is_canceled = the scheduled stop was not served and no replacement ran
df["is_canceled"] = ~df["ran"] & ~df["has_replacement"]

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 29,976,896 records from 2 files
Files: ['data-2026-09.parquet', 'data-2026-10.parquet']


In [3]:
# Define long-distance train types
long_distance_train_types = ["ICE", "IC", "FLX", "EC"]

# Build a display name (e.g. "ICE 123") from the train type and the Zugnummer (train_number)
df_long_distance = df[df["train_type"].isin(long_distance_train_types)].copy()
df_long_distance["train_name"] = df_long_distance["train_type"] + " " + df_long_distance["train_number"]

# Calculate average delays, cancellation percentages, and sample counts by train
train_stats = (
    df_long_distance.groupby("train_name")
    .agg({"delay_in_min": ["mean", "count"], "is_canceled": "mean"})
    .sort_values(("delay_in_min", "count"), ascending=False)
)

# Flatten column names and rename for clarity
train_stats.columns = ["Durchschnittliche Verspätung [min]", "Anzahl Fahrten", "Ausfallquote [%]"]
train_stats = train_stats.reset_index()
train_stats = train_stats.rename(columns={"train_name": "Zugname"})

# Convert cancellation rate to percentage
train_stats["Ausfallquote [%]"] = (train_stats["Ausfallquote [%]"] * 100).round(2)
train_stats["Durchschnittliche Verspätung [min]"] = train_stats["Durchschnittliche Verspätung [min]"].round(2)

# Filter to trains with at least 100 planned journeys
train_stats = train_stats[train_stats["Anzahl Fahrten"] >= 100]

# Reorder columns
train_stats = train_stats[["Zugname", "Durchschnittliche Verspätung [min]", "Ausfallquote [%]", "Anzahl Fahrten"]]

print(f"Found {len(train_stats)} long-distance trains with at least 100 journeys")

Found 993 long-distance trains with at least 100 journeys


In [4]:
content = cell(
    heading("Welche Fernverkehrszüge haben am meisten Verspätungen?", level=1)
    + paragraph(
        "Viele Fernverkehrszüge haben eine eindeutige Nummer in den Daten und damit lässt sich die "
        "Pünktlichkeit und Ausfallquote aller Fernverbindungen von ICE, IC, FLX (FlixTrain) oder EC (EuroCity) ermitteln. "
        "Die Tabelle zeigt alle Züge mit mindestens 100 geplanten Fahrten. "
        "Züge, deren planmäßige Fahrt durch einen Ersatzzug (t=\"e\") durchgeführt wurde, zählen nicht als ausgefallen."
    )
    + table(train_stats)
)
display(HTML(content))

Zugname ⇕,Durchschnittliche Verspätung [min] ⇕,Ausfallquote [%] ⇕,Anzahl Fahrten ⇕
IC 2046,11.5,0.0,1413
IC 2048,9.81,0.0,1394
IC 2065,13.14,0.0,1392
IC 2049,12.59,0.0,1390
IC 2440,8.87,0.0,1390
IC 2045,21.47,0.0,1389
IC 2047,16.61,0.0,1388
ICE 618,16.28,2.91,1376
IC 2441,11.24,0.0,1342
IC 2044,10.93,0.0,1339


In [5]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/zugverbindung.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))